# 01 — Exploración Inicial: Dataset de Partidos ATP (2000–2026)

Este notebook realiza una exploración inicial del dataset `data/matches.csv`, que contiene ~67.000 partidos de tenis masculino profesional (ATP).

**Objetivos:**
- Cargar el dataset y revisar su estructura
- Identificar tipos de datos, nulos y duplicados
- Obtener estadísticas descriptivas básicas
- Sentar las bases para análisis posteriores

## 1. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Librerías cargadas correctamente.")

## 2. Carga del dataset

In [ ]:
df = pd.read_csv("../data/matches.csv", parse_dates=["Date"])

print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Rango de fechas: {df['Date'].min().date()}  →  {df['Date'].max().date()}")

In [ ]:
df.head(10)

## 3. Estructura y tipos de datos

In [ ]:
df.info()

## 4. Valores nulos y centinelas (-1)

In [ ]:
# Nulos reales
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({"nulos": nulos, "% del total": pct})
resumen_nulos[resumen_nulos["nulos"] > 0]

In [ ]:
# Valores centinela -1 (ausencia de dato en columnas numéricas)
centinelas = {
    "Pts_1":  (df["Pts_1"]  == -1).sum(),
    "Pts_2":  (df["Pts_2"]  == -1).sum(),
    "Odd_1":  (df["Odd_1"]  == -1).sum(),
    "Odd_2":  (df["Odd_2"]  == -1).sum(),
}
pd.DataFrame(centinelas, index=["filas con -1"]).T.assign(
    pct=lambda x: (x["filas con -1"] / len(df) * 100).round(2)
)

## 5. Duplicados

In [ ]:
duplicados = df.duplicated().sum()
print(f"Filas duplicadas exactas: {duplicados}")

# Duplicados por partido (misma fecha + jugadores)
dup_partido = df.duplicated(subset=["Date", "Player_1", "Player_2"]).sum()
print(f"Duplicados por Date+Player_1+Player_2: {dup_partido}")

## 6. Variables categóricas: cardinalidad y distribución

In [ ]:
cats = ["Series", "Court", "Surface", "Round", "Best of"]
for col in cats:
    vc = df[col].value_counts()
    print(f"\n── {col} ({vc.shape[0]} valores únicos) ──")
    print(vc.to_string())

## 7. Estadísticas descriptivas

In [ ]:
# Reemplazar centinelas por NaN para estadísticas reales
df_stats = df.copy()
for col in ["Pts_1", "Pts_2", "Rank_1", "Rank_2"]:
    df_stats[col] = df_stats[col].replace(-1, np.nan)
for col in ["Odd_1", "Odd_2"]:
    df_stats[col] = df_stats[col].replace(-1.0, np.nan)

df_stats[["Rank_1", "Rank_2", "Pts_1", "Pts_2", "Odd_1", "Odd_2"]].describe().round(2)

## 8. Visualizaciones exploratorias

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Exploración inicial — ATP Matches 2000–2026", fontsize=14, fontweight="bold")

# 1. Partidos por año
df["Year"] = df["Date"].dt.year
partidos_year = df.groupby("Year").size()
axes[0, 0].bar(partidos_year.index, partidos_year.values, color="steelblue", edgecolor="white")
axes[0, 0].set_title("Partidos por año")
axes[0, 0].set_xlabel("Año")
axes[0, 0].set_ylabel("Partidos")
axes[0, 0].xaxis.set_major_locator(ticker.MultipleLocator(4))

# 2. Distribución por superficie
sup_counts = df["Surface"].value_counts()
colors = ["#e07b39", "#4caf50", "#5b9bd5", "#9b59b6"]
axes[0, 1].pie(sup_counts.values, labels=sup_counts.index, autopct="%1.1f%%",
               colors=colors, startangle=140)
axes[0, 1].set_title("Partidos por superficie")

# 3. Distribución por categoría de torneo
series_counts = df["Series"].value_counts()
axes[1, 0].barh(series_counts.index, series_counts.values, color="coral", edgecolor="white")
axes[1, 0].set_title("Partidos por categoría (Series)")
axes[1, 0].set_xlabel("Partidos")

# 4. Distribución de cuotas (Odd_1) — solo partidos con dato
odds = df_stats["Odd_1"].dropna()
axes[1, 1].hist(odds, bins=50, color="mediumseagreen", edgecolor="white")
axes[1, 1].set_title("Distribución de cuota Odd_1\n(favorito implícito = cuota < 2.0)")
axes[1, 1].set_xlabel("Odd_1")
axes[1, 1].set_ylabel("Frecuencia")
axes[1, 1].axvline(2.0, color="red", linestyle="--", linewidth=1, label="Cuota = 2.0")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 9. Jugadores con más partidos

In [ ]:
# Contar apariciones totales como Player_1 o Player_2
apariciones = pd.concat([df["Player_1"], df["Player_2"]]).value_counts()
top20 = apariciones.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20.index[::-1], top20.values[::-1], color="steelblue", edgecolor="white")
ax.set_title("Top 20 jugadores por partidos disputados (2000–2026)")
ax.set_xlabel("Partidos")
plt.tight_layout()
plt.show()

## 10. Tasa de victorias del favorito (según cuotas)

In [ ]:
# Solo partidos con cuotas disponibles
df_odds = df_stats[df_stats["Odd_1"].notna() & df_stats["Odd_2"].notna()].copy()

# Favorito = jugador con cuota más baja
df_odds["favorito"] = df_odds.apply(
    lambda r: r["Player_1"] if r["Odd_1"] <= r["Odd_2"] else r["Player_2"], axis=1
)
df_odds["favorito_gano"] = df_odds["favorito"] == df_odds["Winner"]

tasa_global = df_odds["favorito_gano"].mean()
print(f"Partidos con cuotas: {len(df_odds):,}")
print(f"Tasa de victoria del favorito: {tasa_global:.1%}")

# Por superficie
tasa_sup = df_odds.groupby("Surface")["favorito_gano"].mean().sort_values(ascending=False)
print("\nPor superficie:")
print(tasa_sup.map("{:.1%}".format).to_string())

## 11. Resumen y próximos pasos

### Hallazgos clave

| Variable | Observación |
|---|---|
| **Nulos reales** | Muy pocos o ninguno en columnas críticas |
| **Centinelas (-1)** | `Pts` y `Odds` ausentes en partidos ~2000–2005 |
| **Superficie** | Hard domina (~55%), seguido de Clay (~30%) |
| **Categorías** | ATP250/International son la mayoría de los partidos |
| **Favorito gana** | ~65–68% de los partidos (baseline para modelos) |
| **Jugadores** | Federer, Nadal y Djokovic lideran en partidos disputados |

### Próximos notebooks sugeridos

- `02_jugadores.ipynb` — Rankings históricos, evolución de jugadores top, head-to-head
- `03_superficies.ipynb` — Análisis de rendimiento por superficie
- `04_upsets.ipynb` — Análisis de sorpresas según cuotas y rankings
- `05_modelo_predictivo.ipynb` — Features engineering y modelo de clasificación (ganador)